# 03 — Fit and tune the two models

**Stage:** modelling.

This notebook:
1. chooses the columns the models may use, and says why some are left out;
2. tunes a **logistic regression** (the baseline) and a **random forest** (the flexible model) on the validation period;
3. fits the chosen models on the training period only, for two feature sets: with and without the economic context;
4. saves every model's predicted probability for every call, so that notebook 04 can evaluate them period by period.

Nothing here looks at the test periods: they are used only in notebook 04.

> **Origin.** This notebook was generated by Claude (Anthropic), a generative AI assistant, at the student's request; the modelling choices were discussed with the student. See "Declaration of tools" in `README.md`.

In [1]:
import sys
from pathlib import Path

# Make the shared modules in src/ importable from this notebook
sys.path.append(str(Path.cwd().parent))

In [2]:
import json

import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from src.data_utils import load_processed, PROCESSED_FILE, RESULTS_DIR, MACRO_COLUMNS
from src.modelling import (FEATURE_SETS, MODEL_NAMES, EXCLUDED, make_model,
                           feature_columns, target_vector)

df = load_processed()
RESULTS_DIR.mkdir(exist_ok=True)
df.shape

(41188, 24)

## 1. Which columns the models use

The list is defined once, in `src/modelling.py`. Three decisions:

| Columns | Decision | Reason |
|---|---|---|
| `previous`, `pdays`, `poutcome` (earlier campaigns) | left out | In the training period nobody had been contacted before, so these columns are constant: there is nothing to learn from them. |
| `month` | left out | The training period only contains May to August. The test periods contain other months (March, October, …), which the model would never have seen. Here the month mostly tells *which period* a call belongs to, not a seasonal effect. |
| the five economic columns (`euribor3m`, …) | **kept** in the main models, removed in a second version | They are the heart of the question: does a model trained before the crisis survive the change of context? The version without them is a robustness check. |

`duration` was already removed in notebook 01. The cell below checks the first two decisions on the data.

In [3]:
train = df[df["period"] == "train"]

for col in ["previous", "pdays", "poutcome"]:
    print(f"{col}: values in training = {sorted(train[col].unique())}")

print("Months in training:", sorted(train["month"].unique()))
print("Months in the other periods:", sorted(df.loc[df["period"] != "train", "month"].unique()))

previous: values in training = [np.int64(0)]
pdays: values in training = [np.int64(999)]
poutcome: values in training = ['nonexistent']
Months in training: ['aug', 'jul', 'jun', 'may']
Months in the other periods: ['apr', 'aug', 'dec', 'jul', 'jun', 'mar', 'may', 'nov', 'oct', 'sep']


In [4]:
for name, cols in FEATURE_SETS.items():
    print(f"{name}: {len(cols['categorical'])} categorical + {len(cols['numeric'])} numeric columns")
    print("   ", feature_columns(name))

with_macro: 8 categorical + 7 numeric columns
    ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'day_of_week', 'age', 'campaign', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
without_macro: 8 categorical + 2 numeric columns
    ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'day_of_week', 'age', 'campaign']


## 2. Training and validation data

The models are fitted on the training period (May–Aug 2008) and their settings are chosen on the validation period (Oct–Dec 2008). The target is 1 if the client subscribed, 0 otherwise.

In [5]:
valid = df[df["period"] == "validation"]
y_train, y_valid = target_vector(train), target_vector(valid)

pd.DataFrame({"calls": [len(train), len(valid)],
              "subscription_rate": [y_train.mean(), y_valid.mean()]},
             index=["train", "validation"]).round(3)

,calls,subscription_rate
train,23997,0.046
validation,3693,0.063


## 3. How the settings are chosen

Each model has settings that are not learned from the data:
- **logistic regression:** `C`, the strength of the penalty on large coefficients (small `C` = strong penalty, simpler model);
- **random forest:** `max_depth`, how deep each tree may grow, and `min_samples_leaf`, the smallest group of calls a tree may end in (both limit how closely the forest follows the training data).

For each candidate, the model is fitted on the training period and scored on the validation period. The setting with the highest **ROC-AUC** on validation is kept. ROC-AUC measures how well the model *ranks* clients, from most to least likely to subscribe, and does not depend on the share of clients who subscribe, which changes a lot over time. The precision-recall AUC (PR-AUC) is reported as well.

The grids are small on purpose: the project must run in the course workspace, and a few well-spaced values are enough to find the right order of magnitude.

In [6]:
GRIDS = {
    "logistic_regression": [{"C": c} for c in [0.001, 0.01, 0.1, 1, 10]],
    "random_forest": [{"max_depth": d, "min_samples_leaf": leaf}
                      for d in [4, 8, 12, None] for leaf in [1, 10, 50]],
}


def score(model, data, y):
    proba = model.predict_proba(data)[:, 1]
    return roc_auc_score(y, proba), average_precision_score(y, proba)


rows = []
for feature_set in FEATURE_SETS:
    cols = feature_columns(feature_set)
    for model_name in MODEL_NAMES:
        for params in GRIDS[model_name]:
            model = make_model(model_name, feature_set, **params)
            model.fit(train[cols], y_train)
            roc, pr = score(model, valid[cols], y_valid)
            rows.append({"feature_set": feature_set, "model": model_name,
                         "params": json.dumps(params), "valid_roc_auc": roc, "valid_pr_auc": pr})

tuning = pd.DataFrame(rows)
tuning.round(3)

,feature_set,model,params,valid_roc_auc,valid_pr_auc
0,with_macro,logistic_regression,"{""C"": 0.001}",0.612,0.220
1,with_macro,logistic_regression,"{""C"": 0.01}",0.513,0.064
2,with_macro,logistic_regression,"{""C"": 0.1}",0.432,0.056
3,with_macro,logistic_regression,"{""C"": 1}",0.434,0.057
4,with_macro,logistic_regression,"{""C"": 10}",0.434,0.057
5,with_macro,random_forest,"{""max_depth"": 4, ""min_samples_leaf"": 1}",0.431,0.055
6,with_macro,random_forest,"{""max_depth"": 4, ""min_samples_leaf"": 10}",0.428,0.055
7,with_macro,random_forest,"{""max_depth"": 4, ""min_samples_leaf"": 50}",0.432,0.057
8,with_macro,random_forest,"{""max_depth"": 8, ""min_samples_leaf"": 1}",0.427,0.054
9,with_macro,random_forest,"{""max_depth"": 8, ""min_samples_leaf"": 10}",0.415,0.053


## 4. Table 3: the chosen settings

For each model and feature set, the row with the best validation ROC-AUC.

In [7]:
best = (tuning.sort_values("valid_roc_auc", ascending=False)
              .groupby(["feature_set", "model"], sort=False).head(1)
              .sort_values(["feature_set", "model"])
              .reset_index(drop=True))
best.to_csv(RESULTS_DIR / "table3_validation.csv", index=False)
best.round(3)

,feature_set,model,params,valid_roc_auc,valid_pr_auc
0,with_macro,logistic_regression,"{""C"": 0.001}",0.612,0.220
1,with_macro,random_forest,"{""max_depth"": 12, ""min_samples_leaf"": 1}",0.461,0.058
2,without_macro,logistic_regression,"{""C"": 0.001}",0.446,0.063
3,without_macro,random_forest,"{""max_depth"": null, ""min_samples_leaf"": 1}",0.449,0.061


*Write your own reading of Table 3 here:* which model does better on validation, and does removing the economic columns change much at this stage?

## 5. Fit the chosen models and save their predictions

Each chosen model is fitted once more on the training period (the validation period is used only to choose settings), then gives a predicted probability of subscribing for **every** call in the data. These probabilities are saved to `data/processed/predictions.csv`; notebook 04 compares them with what actually happened, period by period.

In [8]:
predictions = df[["year_month", "period"]].copy()
predictions["y"] = target_vector(df)

chosen = {}
for _, row in best.iterrows():
    params = json.loads(row["params"])
    cols = feature_columns(row["feature_set"])
    model = make_model(row["model"], row["feature_set"], **params)
    model.fit(train[cols], y_train)
    column = f"{row['model']}__{row['feature_set']}"
    predictions[column] = model.predict_proba(df[cols])[:, 1]
    chosen[column] = params

predictions.to_csv(PROCESSED_FILE.parent / "predictions.csv", index=False)
with open(RESULTS_DIR / "chosen_settings.json", "w") as f:
    json.dump(chosen, f, indent=2)

print("Saved predictions for", len(predictions), "calls and", len(chosen), "models")
predictions.head()

Saved predictions for 41188 calls and 4 models


,year_month,period,y,logistic_regression__with_macro,random_forest__with_macro,logistic_regression__without_macro,random_forest__without_macro
0,2008-05,train,0,0.032053,0.016427,0.040006,0.000000
1,2008-05,train,0,0.032486,0.021507,0.040302,0.006667
2,2008-05,train,0,0.034475,0.021143,0.043196,0.003333
3,2008-05,train,0,0.034403,0.015698,0.043052,0.000000
4,2008-05,train,0,0.032948,0.016938,0.041120,0.000000


**Quick check.** On the training and validation periods, the predicted probabilities should be close on average to the real subscription rate. Notebook 04 does the real evaluation on the test periods.

In [9]:
model_columns = list(chosen)
(predictions[predictions["period"].isin(["train", "validation"])]
   .groupby("period")[["y"] + model_columns].mean()
   .loc[["train", "validation"]]
   .round(3))

,y,logistic_regression__with_macro,random_forest__with_macro,logistic_regression__without_macro,random_forest__without_macro
period,,,,,
train,0.046,0.046,0.046,0.046,0.047
validation,0.063,0.015,0.052,0.050,0.061
